Imports

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import copy
from torchsummary import summary

from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
import numpy as np
from sklearn.metrics import accuracy_score

In [13]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print(f'Using GPU: {torch.cuda.get_device_name(0)}')
else:
    print('Using CPU')

Using GPU: NVIDIA GeForce RTX 4060 Laptop GPU


Dataset

In [14]:
train_dir = r"C:\SUTD\50.021 Artificial Intelligence\pppppp\Master_Dataset\train"
val_dir = r"C:\SUTD\50.021 Artificial Intelligence\pppppp\Master_Dataset\val"
test_dir = r"C:\SUTD\50.021 Artificial Intelligence\pppppp\Master_Dataset\test"

In [15]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [16]:
# Load datasets
train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)
test_dataset = datasets.ImageFolder(test_dir, transform=transform)

print("Classes:", train_dataset.classes)
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print("Train class_to_idx:", train_dataset.class_to_idx)
print("Val class_to_idx:", val_dataset.class_to_idx)
print("Test class_to_idx:", test_dataset.class_to_idx)

Classes: ['NORMAL', 'PNEUMONIA']
Train size: 14054
Val size: 1757
Test size: 1757
Train class_to_idx: {'NORMAL': 0, 'PNEUMONIA': 1}
Val class_to_idx: {'NORMAL': 0, 'PNEUMONIA': 1}
Test class_to_idx: {'NORMAL': 0, 'PNEUMONIA': 1}


Load Model Function

In [17]:
def load_model(model_name, path, device):
    if model_name == "alexnet":
        model = build_alexnet()
    elif model_name == "resnet18":
        model = build_resnet18()
    else:
        raise ValueError("Unknown model name")

    state_dict = torch.load(path, map_location=device)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    return model

Training loop and fgsm

In [18]:

def fgsm_attack(x, epsilon, grad, x_min, x_max):
    x_adv = x + epsilon * grad.sign()
    x_adv = torch.max(torch.min(x_adv, x_max), x_min)
    return x_adv.detach()

def train_adversarial_fgsm(model, train_loader, val_loader, device, epochs=10, epsilon=0.01, lr=1e-4):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

    x_min = (0.0 - mean) / std
    x_max = (1.0 - mean) / std

    for epoch in range(epochs):
        model.train()
        running_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            # Step 1: get gradient wrt input
            images_for_attack = images.clone().detach().requires_grad_(True)
            outputs = model(images_for_attack)
            clean_loss = criterion(outputs, labels)

            optimizer.zero_grad()
            clean_loss.backward()

            # Step 2: generate adversarial images
            adv_images = fgsm_attack(
                images_for_attack,
                epsilon,
                images_for_attack.grad,
                x_min,
                x_max
            )

            # Step 3: train on clean + adversarial
            optimizer.zero_grad()

            clean_outputs = model(images)
            adv_outputs = model(adv_images)

            clean_loss = criterion(clean_outputs, labels)
            adv_loss = criterion(adv_outputs, labels)
            loss = 0.5 * clean_loss + 0.5 * adv_loss

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            preds = adv_outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_acc = 100 * correct / total
        avg_loss = running_loss / len(train_loader)

        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {avg_loss:.4f} | Adv-train acc: {train_acc:.2f}%")

        if val_loader is not None:
            evaluate_fgsm(model, val_loader, epsilon, device, x_min, x_max)

# --- Evaluation loop ---
def evaluate_fgsm(model, loader, epsilon, device, x_min, x_max):
    model.eval()
    loss_fn = nn.CrossEntropyLoss()
    correct_clean = 0
    correct_adv   = 0
    total         = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        images = images.clone().detach().requires_grad_(True)

        

        # Clean accuracy
        out = model(images)
        correct_clean += (out.argmax(1) == labels).sum().item()

        # Compute gradient w.r.t. input
        loss = loss_fn(out, labels) # FIX: Use out.logits here
        model.zero_grad()
        loss.backward()

        

        # Perturb and re-evaluate
        adv_images = fgsm_attack(images, epsilon, images.grad.detach(), x_min, x_max)
        with torch.no_grad():
            adv_out = model(adv_images)
        correct_adv += (adv_out.argmax(1) == labels).sum().item()

        total += labels.size(0)

    clean_acc = 100 * correct_clean / total
    adv_acc   = 100 * correct_adv   / total
    print(f"ε={epsilon:.3f} | Clean acc: {clean_acc:.1f}%  Adv acc: {adv_acc:.1f}%  "
          f"Drop: {clean_acc - adv_acc:.1f}%")
    return clean_acc, adv_acc

AlexNET

In [19]:
def build_alexnet():
    model = models.alexnet(weights=None)
    model.classifier = nn.Sequential(
        nn.Dropout(),
        nn.Linear(9216, 4096),
        nn.ReLU(inplace=True),
        nn.Dropout(),
        nn.Linear(4096, 1024),
        nn.ReLU(inplace=True),
        nn.Linear(1024, 512),
        nn.ReLU(inplace=True),
        nn.Linear(512, 128),
        nn.ReLU(inplace=True),
        nn.Linear(128, 32),
        nn.ReLU(inplace=True),
        nn.Linear(32, 2)
    )
    return model

In [20]:
alexnet_model = load_model("alexnet", "alexnet_finetuned_baseline.pth", device)

train_adversarial_fgsm(
    model=alexnet_model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=8,
    epsilon=0.01,
    lr=1e-4
)

C:\Users\thoai\AppData\Local\Temp\ipykernel_33676\4156658993.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path, map_location=device)


Epoch [1/8] | Loss: 0.5660 | Adv-train acc: 64.20%
ε=0.010 | Clean acc: 78.8%  Adv acc: 69.7%  Drop: 9.2%
Epoch [2/8] | Loss: 0.4980 | Adv-train acc: 71.94%
ε=0.010 | Clean acc: 79.4%  Adv acc: 72.2%  Drop: 7.2%
Epoch [3/8] | Loss: 0.4716 | Adv-train acc: 73.64%
ε=0.010 | Clean acc: 79.4%  Adv acc: 73.3%  Drop: 6.1%
Epoch [4/8] | Loss: 0.4583 | Adv-train acc: 75.14%
ε=0.010 | Clean acc: 79.9%  Adv acc: 74.7%  Drop: 5.2%
Epoch [5/8] | Loss: 0.4387 | Adv-train acc: 76.22%
ε=0.010 | Clean acc: 78.3%  Adv acc: 72.3%  Drop: 6.0%
Epoch [6/8] | Loss: 0.4221 | Adv-train acc: 76.97%
ε=0.010 | Clean acc: 80.1%  Adv acc: 73.1%  Drop: 6.9%
Epoch [7/8] | Loss: 0.4043 | Adv-train acc: 77.94%
ε=0.010 | Clean acc: 78.7%  Adv acc: 72.2%  Drop: 6.5%
Epoch [8/8] | Loss: 0.3757 | Adv-train acc: 79.14%
ε=0.010 | Clean acc: 79.4%  Adv acc: 70.9%  Drop: 8.5%


resnet

In [21]:
def build_resnet18():
    model = models.resnet18(weights=None)
    model.fc = nn.Sequential(
        nn.Dropout(),
        nn.Linear(512, 128),
        nn.ReLU(inplace=True),
        nn.Linear(128, 32),
        nn.ReLU(inplace=True),
        nn.Linear(32, 2),
        nn.ReLU(inplace=True)
    )
    return model

In [ ]:
resnet_model = load_model("resnet18", "resnet18_finetuned_baseline.pth", device)

train_adversarial_fgsm(
    model=resnet_model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=8,
    epsilon=0.01,
    lr=1e-4
)

C:\Users\thoai\AppData\Local\Temp\ipykernel_33676\4156658993.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path, map_location=device)


Epoch [1/8] | Loss: 0.5290 | Adv-train acc: 68.83%
ε=0.010 | Clean acc: 80.2%  Adv acc: 71.7%  Drop: 8.5%
Epoch [2/8] | Loss: 0.4569 | Adv-train acc: 75.66%
ε=0.010 | Clean acc: 80.0%  Adv acc: 74.6%  Drop: 5.4%
Epoch [3/8] | Loss: 0.4281 | Adv-train acc: 77.54%
ε=0.010 | Clean acc: 81.5%  Adv acc: 75.0%  Drop: 6.5%
Epoch [4/8] | Loss: 0.4026 | Adv-train acc: 79.09%
ε=0.010 | Clean acc: 80.4%  Adv acc: 74.4%  Drop: 6.0%
Epoch [5/8] | Loss: 0.3735 | Adv-train acc: 80.16%
ε=0.010 | Clean acc: 82.3%  Adv acc: 74.9%  Drop: 7.4%
Epoch [6/8] | Loss: 0.3317 | Adv-train acc: 82.47%
ε=0.010 | Clean acc: 81.7%  Adv acc: 72.7%  Drop: 9.0%
Epoch [7/8] | Loss: 0.2653 | Adv-train acc: 86.57%
ε=0.010 | Clean acc: 79.7%  Adv acc: 69.1%  Drop: 10.6%
Epoch [8/8] | Loss: 0.2336 | Adv-train acc: 85.83%
ε=0.010 | Clean acc: 80.2%  Adv acc: 71.1%  Drop: 9.1%


: 

In [3]:
model = tf.keras.models.load_model("baseline_resnet152v2_adversarial_finetunedonmaster.keras")
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet152v2 (Functional)        │ (None, 7, 7, 2048)     │    58,331,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 68,056,325 (259.61 MB)

 Trainable params: 4,731,137 (18.05 MB)

 Non-trainable params: 53,862,912 (205.47 MB)

 Optimizer params: 9,462,276 (36.10 MB)

In [4]:
IMG_SIZE = 224
BATCH = 16
SEED = 42

train_val_datagen = ImageDataGenerator(
    rescale=1/255.,
    validation_split=0.2
)

ds_master_train = train_val_datagen.flow_from_directory(
    r"Master_Dataset/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=True,
    subset="training",
    seed=SEED
)

ds_master_val = train_val_datagen.flow_from_directory(
    r"Master_Dataset/val",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=False,
    subset="validation",
    seed=SEED
)

test_datagen = ImageDataGenerator(rescale=1/255.)

ds_master_test = test_datagen.flow_from_directory(
    r"Master_Dataset/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=False
)

Found 11244 images belonging to 2 classes.
Found 350 images belonging to 2 classes.
Found 1757 images belonging to 2 classes.


In [10]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

y_true = []
y_pred = []

for images, labels in ds_master_test:
    preds = model.predict(images, verbose=0)
    preds = (preds > 0.5).astype(int).flatten()

    y_true.extend(labels.flatten())
    y_pred.extend(preds)

y_true = np.array(y_true).astype(int)
y_pred = np.array(y_pred).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=4))

KeyboardInterrupt: 